# DICOM Mock Server - Quickstart & Integration Guide

Welcome to the **GoSmart.Health DICOM Mock Server** quickstart notebook!

This interactive guide demonstrates how to communicate with the mock server across multiple protocols:
1. **REST API**: Health checks, server configuration, and Modality Worklist (MWL) management.
2. **DICOMweb (PS 3.18)**: Study discovery via **QIDO-RS** and rendered frame retrieval via **WADO-RS**.
3. **DIMSE Network Communication**: Verification service (**C-ECHO**) via `pynetdicom`.

> **Prerequisite**: Ensure the mock server is running locally (e.g. `./start.sh` or `docker compose up`) before executing these cells.

In [ ]:
import io
import json

import httpx
from PIL import Image
from pynetdicom import AE
from pynetdicom.sop_class import Verification

# Server configuration defaults
BASE_URL = "http://127.0.0.1:8000"
SCP_HOST = "127.0.0.1"
SCP_PORT = 11112
SCP_AE = "MOCK_SCP"

client = httpx.Client(base_url=BASE_URL, timeout=10.0)
print("Configured client pointing to:", BASE_URL)

## 1. Mock Server Health & Status Check

Let's verify that the mock server's HTTP REST endpoints and DICOM SCP service are up and healthy.

In [ ]:
# 1. Application Health Check
health_res = client.get("/api/v1/health")
print(f"GET /api/v1/health -> HTTP {health_res.status_code}")
print(json.dumps(health_res.json(), indent=2))

# 2. DICOM SCP Service Status
scp_res = client.get("/api/v1/scp/status")
print(f"\nGET /api/v1/scp/status -> HTTP {scp_res.status_code}")
print(json.dumps(scp_res.json(), indent=2))

## 2. Modality Worklist (MWL) Management

The server maintains an in-memory active worklist simulating scheduled patient procedures within a retention window.

In [ ]:
# Query active MWL items
mwl_res = client.get("/api/v1/mwl")
mwl_entries = mwl_res.json()
print(f"Total active MWL entries: {len(mwl_entries)}")

if mwl_entries:
    print("\nFirst active worklist entry sample:")
    print(json.dumps(mwl_entries[0], indent=2))
else:
    print("No worklist entries currently active.")

### Scheduling a New Procedure Order via REST API

Let's schedule a dedicated mock examination for testing.

In [ ]:
order_payload = {
    "patient_name": "NOTEBOOK^DEMO^JOHN",
    "patient_id": "NB-PAT-001",
    "accession_number": "NB-ACC-001",
    "modality": "CT",
    "study_description": "CT Chest with Contrast - Quickstart Notebook Demo",
    "num_instances": 4,
}

order_res = client.post("/api/v1/mwl/generate", json=order_payload)
print(f"POST /api/v1/mwl/generate -> HTTP {order_res.status_code}")
created_entry = order_res.json()
print(json.dumps(created_entry, indent=2))

## 3. DICOMweb QIDO-RS Study Search

The server supports standard DICOMweb **QIDO-RS** queries (`GET /dicomweb/studies`).
Let's query for the study matching our newly scheduled patient ID.

In [ ]:
qido_res = client.get("/dicomweb/studies", params={"PatientID": "NB-PAT-001"})
print(f"GET /dicomweb/studies?PatientID=NB-PAT-001 -> HTTP {qido_res.status_code}")

studies = qido_res.json()
print(f"Found {len(studies)} matching study/studies.\n")

study_instance_uid = None
if studies:
    first_study = studies[0]
    study_instance_uid = first_study.get("0020000D", {}).get("Value", [None])[0]
    patient_name = first_study.get("00100010", {}).get("Value", [{}])[0].get("Alphabetic", "Unknown")
    study_desc = first_study.get("00081030", {}).get("Value", ["Unknown"])[0]

    print(f"StudyInstanceUID: {study_instance_uid}")
    print(f"PatientName:      {patient_name}")
    print(f"StudyDescription: {study_desc}")

## 4. DICOMweb WADO-RS: Retrieve Series, Instances & Rendered Frame

Now let's inspect the series and instances within this study, and fetch an 8-bit rendered JPEG/PNG view using WADO-RS (`.../rendered`).

In [ ]:
if study_instance_uid:
    # 1. Query series in study
    series_res = client.get(f"/dicomweb/studies/{study_instance_uid}/series")
    series_list = series_res.json()
    print(f"Series in study: {len(series_list)}")
    series_uid = series_list[0].get("0020000E", {}).get("Value", [None])[0]
    print(f"First SeriesInstanceUID: {series_uid}")

    # 2. Query instances in series
    instances_res = client.get(f"/dicomweb/studies/{study_instance_uid}/series/{series_uid}/instances")
    instance_list = instances_res.json()
    print(f"Instances in series: {len(instance_list)}")
    sop_instance_uid = instance_list[0].get("00080018", {}).get("Value", [None])[0]
    print(f"First SOPInstanceUID: {sop_instance_uid}")

    # 3. Retrieve rendered 8-bit view via WADO-RS
    rendered_path = f"/dicomweb/studies/{study_instance_uid}/series/{series_uid}/instances/{sop_instance_uid}/rendered"
    rendered_res = client.get(rendered_path)
    print(f"\nGET {rendered_path} -> HTTP {rendered_res.status_code}")
    print(f"Content-Type: {rendered_res.headers.get('content-type')}")

    if rendered_res.status_code == 200:
        img = Image.open(io.BytesIO(rendered_res.content))
        print(f"Image format: {img.format}, Dimensions: {img.size}, Mode: {img.mode}")
        display(img)
else:
    print("No study instance available to retrieve.")

## 5. DIMSE Network Communication (C-ECHO)

The mock server runs a standard DICOM Upper Layer Protocol listener on TCP Port 11112.
Let's perform a Verification SCU (**C-ECHO**) request using `pynetdicom`.

In [ ]:
ae = AE(ae_title="NOTEBOOK_SCU")
ae.add_requested_context(Verification)

print(f"Connecting to SCP AE '{SCP_AE}' at {SCP_HOST}:{SCP_PORT}...")
assoc = ae.associate(SCP_HOST, SCP_PORT, ae_title=SCP_AE)

if assoc.is_established:
    print("Association successfully established!")
    status = assoc.send_c_echo()
    if status:
        print(f"C-ECHO Response Status: 0x{status.Status:04X} (Success = 0x0000)")
    else:
        print("C-ECHO request timed out or received invalid response.")
    assoc.release()
    print("Association released.")
else:
    print(f"Failed to establish association with {SCP_AE}@{SCP_HOST}:{SCP_PORT}.")

## Summary

You have successfully verified:
- Server health and SCP status via REST API.
- Scheduling simulated patient exams via Modality Worklist (MWL).
- Querying and discovering studies via DICOMweb QIDO-RS.
- Retrieving rendered 8-bit image frames via DICOMweb WADO-RS.
- Establishing DIMSE associations and verifying connectivity via C-ECHO.

For additional API endpoints, interactive OpenAPI schemas, and configuration options:
- Interactive Swagger UI: [http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs)
- ReDoc Reference: [http://127.0.0.1:8000/redoc](http://127.0.0.1:8000/redoc)